<a href="https://colab.research.google.com/github/amcmdv/special-gpt-guide/blob/main/04052026_GRC_%26_Identity_Intelligence_PoC_(Demo).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **GRC & Identity Intelligence PoC**

This script provides a real-time dashboard for monitoring **Governance, Risk, and Compliance (GRC)** and **Identity Access Management (IAM)**.

---

### **Key Improvements for GitHub Hosting**

* **Modular Architecture:** I separated the `IAMEngine` (Backend) from the Gradio `demo` (Frontend). This makes it easier for others to swap the mock data for a real API.
* **Threaded Simulation:** The background events run in a `daemon` thread. This prevents the dashboard from freezing while waiting for data.
* **Visual Polish:** Switched to the `Soft()` theme and added a dark-themed knowledge graph for a more modern "Security Operations Center" (SOC) feel.
* **File Management:** The knowledge graph is saved and read locally, ensuring compatibility across different OS environments.
* **Reduced Complexity:** Removed 150+ lines of redundant pandas operations while keeping the "core magic" (Risk scores, SoD conflicts, and Graph views).


In [3]:
# [SECTION 1] Install Dependencies
# We upgrade to Gradio 5, which plays nicely with Colab's native libraries.
# We explicitly upgrade websockets to satisfy the google-adk/yfinance requirements.
!pip -q install --upgrade gradio>=5.0 pyvis websockets>=15.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 16.0 which is incompatible.


In [4]:
import pandas as pd
import plotly.express as px
import networkx as nx
from pyvis.network import Network
import gradio as gr
import random
import uuid
import threading
import time
from datetime import datetime, timezone

# --- 1. MOCK DATA GENERATOR (The "Engine") ---
class IAMEngine:
    def __init__(self, n_users=100):
        self.depts = ["Finance", "Engineering", "Sales", "Security"]
        self.apps = ["GitHub", "Okta", "AWS", "Workday"]
        self.users = pd.DataFrame([{
            "user_id": str(uuid.uuid4())[:8],
            "name": f"Employee {i}",
            "dept": random.choice(self.depts),
            "risk_score": random.randint(10, 50)
        } for i in range(n_users)])

        self.alerts = pd.DataFrame(columns=["timestamp", "type", "severity", "user"])
        self.lock = threading.Lock()

    def simulate_risk_event(self):
        """Simulates incoming security events and recalculates risk."""
        with self.lock:
            idx = random.randint(0, len(self.users) - 1)
            # Simulate a risk spike (e.g., failed login or SoD conflict)
            self.users.at[idx, 'risk_score'] = min(100, self.users.at[idx, 'risk_score'] + random.randint(5, 20))

            new_alert = {
                "timestamp": datetime.now().strftime("%H:%M:%S"),
                "type": random.choice(["SoD Conflict", "Anomalous Login", "Privilege Escalation"]),
                "severity": random.choice(["Medium", "High", "Critical"]),
                "user": self.users.at[idx, 'name']
            }
            self.alerts = pd.concat([pd.DataFrame([new_alert]), self.alerts]).head(10)

# --- 2. ANALYTICS & VISUALIZATION ---
engine = IAMEngine()

def get_dashboard_data():
    with engine.lock:
        # KPI 1: Risk Distribution
        fig_risk = px.histogram(engine.users, x="risk_score", color="dept", title="Risk Distribution by Dept")

        # KPI 2: Top Offenders
        top_risks = engine.users.sort_values("risk_score", ascending=False).head(10)

        # KPI 3: Knowledge Graph (User -> Dept)
        G = nx.Graph()
        for _, row in top_risks.iterrows():
            G.add_node(row['name'], size=row['risk_score'], group=row['dept'])
            G.add_edge(row['name'], row['dept'])

        net = Network(height="400px", width="100%", bgcolor="#222222", font_color="white")
        net.from_nx(G)
        graph_path = "graph.html"
        net.save_graph(graph_path)

        return fig_risk, top_risks[["name", "dept", "risk_score"]], engine.alerts, graph_path

# --- 3. GRADIO INTERFACE ---
with gr.Blocks(theme=gr.themes.Soft(), title="Identity Risk PoC") as demo:
    gr.Markdown("# 🛡️ GRC & Identity Risk Intelligence PoC")

    with gr.Row():
        with gr.Column(scale=2):
            risk_plot = gr.Plot(label="Risk Heatmap")
        with gr.Column(scale=1):
            alert_df = gr.DataFrame(label="Live Security Alerts")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🚩 High Risk Access Queue")
            risk_df = gr.DataFrame()
        with gr.Column():
            gr.Markdown("### 🕸️ Identity Graph")
            graph_html = gr.HTML()

    # Background Simulator
    def run_sim():
        while True:
            engine.simulate_risk_event()
            time.sleep(3)

    threading.Thread(target=run_sim, daemon=True).start()

    # Auto-refresh logic (Every 3 seconds)
    def update_ui():
        f, t, a, g_path = get_dashboard_data()
        with open(g_path, 'r') as f_html:
            html_content = f_html.read()
        return f, t, a, html_content

    dep_trigger = gr.Timer(3)
    dep_trigger.tick(update_ui, outputs=[risk_plot, risk_df, alert_df, graph_html])

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_2530/970316780.py:67: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Identity Risk PoC") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6ef73bc357bebd7583.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
